# 06_add_decoys — decoy(가정 inactive) 생성 [전역창 방식]

**한 줄 요약:** inactive가 부족하니, PubChem 무작위 분자 중 **active와 성질은 비슷(전역 5~95% 범위)하지만 구조는 다른(Tanimoto≤0.35)** 것을 골라 **가정 inactive(decoy)** 로 추가해 active:inactive를 1:1로 맞춘다.
**용어:** decoy=실측이 아니라 '결합 안 할 것으로 가정한' 분자. Tanimoto=지문 유사도(0~1).
**큰 흐름:** ① 준비 → ② active/필요수 계산 → ③ 성질범위·지문 → ④ PubChem 요청 함수 → ⑤ decoy 수집 → ⑥ 저장

> **📌 읽는 법**: 각 코드 셀은 **[① 무슨 작업] → [② 코드] → [③ 🔎 코드 뜯어보기]**. ③은 그 셀에 **처음 나온** 함수·문법(기초 반복은 *(01에서 설명)*).

### 준비 — 폴더 위치 맞추기
어느 폴더에서 열어도 프로젝트 최상위에서 실행되도록 이동.

In [ ]:
# 노트북을 어느 폴더에서 열든 프로젝트 루트에서 실행되도록 이동
import os
if not os.path.isdir('data') and os.path.basename(os.getcwd()) in ('notebooks', 'scripts'):
    os.chdir('..')
print('작업 폴더:', os.getcwd())

🔎 *(01에서 설명)*: `os.chdir('..')`=상위 폴더 이동, `print`=출력.

### 셀 1 — 도구 불러오기 + 설정
웹 요청(requests) 등 라이브러리와 임계값·범위 상수를 준비한다.

In [ ]:
import time
import random
import numpy as np
import pandas as pd
import requests
from rdkit import Chem, DataStructs
from rdkit.Chem import Descriptors, rdMolDescriptors, rdFingerprintGenerator
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")

SRC = "data/HSD17B13_IC50_merged.xlsx"
OUT_XLSX = "data/HSD17B13_train_with_decoys.xlsx"
OUT_CSV = "data/HSD17B13_decoys.csv"
ACTIVE_MAX = 10000.0
TANIMOTO_MAX = 0.35
MAX_CID = 170_000_000
BATCH = 150
SEED = 42
BASE = "https://pubchem.ncbi.nlm.nih.gov/rest/pug"
MAX_SAMPLED = 400_000   # 무한루프 방지: 이만큼 CID 살펴봐도 못 채우면 중단

random.seed(SEED)
np.random.seed(SEED)

🔎 **코드 뜯어보기 (셀 1)**
- `import requests` : 인터넷에 **요청을 보내 데이터를 받는** 라이브러리(PubChem 접속용). `import random`=난수, `import time`=대기.
- `TANIMOTO_MAX = 0.35` 등 : decoy 조건 상수(구조 유사도 상한 등).

### 셀 2 — 라벨링 → active/inactive 목록 + 필요한 decoy 수
데이터를 라벨링해 active·inactive 목록을 만들고, 1:1을 맞추려면 decoy가 몇 개 필요한지 계산한다.

In [ ]:
def make_label(ic50, rel):
    rel = str(rel).strip()
    if pd.isna(ic50):
        return np.nan
    if rel in ("<", "<="):
        return 1 if ic50 <= ACTIVE_MAX else 0
    if rel in (">", ">="):
        return 0
    return 1 if ic50 <= ACTIVE_MAX else 0


df = pd.read_excel(SRC, sheet_name="same_dedup_keepdiff")
df = df.dropna(subset=["canonical_smiles", "ic50_nM"]).copy()
df["label"] = [make_label(v, r) for v, r in zip(df["ic50_nM"], df["relation"])]
df = df.dropna(subset=["label"])
df["label"] = df["label"].astype(int)
comp = df.groupby("canonical_smiles")["label"].max().reset_index()

actives = comp[comp.label == 1]["canonical_smiles"].tolist()
inactives = comp[comp.label == 0]["canonical_smiles"].tolist()
known = set(actives) | set(inactives)
n_need = max(0, len(actives) - len(inactives))
print(f"active {len(actives)} / inactive(실측) {len(inactives)} → 필요한 decoy {n_need}개 (목표 1:1)")

🔎 **코드 뜯어보기 (셀 2)** *(make_label은 05에서 설명)*
- `known = set(actives) | set(inactives)` : **set**=중복 없는 집합, `|`=합집합(둘을 합침). '이미 아는 분자' 모음.
- `n_need = max(0, len(actives) - len(inactives))` : 필요한 decoy 수 = active 수 − 실측 inactive 수(음수면 0).

### 셀 3 — active의 성질 범위(5~95%) + 지문
decoy를 고를 기준인 active의 성질(분자량·logP 등) 범위와 지문을 미리 계산한다.

In [ ]:
gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)


def props(mol):
    return (Descriptors.MolWt(mol),
            Descriptors.MolLogP(mol),
            rdMolDescriptors.CalcNumHBD(mol),
            rdMolDescriptors.CalcNumHBA(mol),
            rdMolDescriptors.CalcNumRotatableBonds(mol),
            Chem.GetFormalCharge(mol))


active_mols, active_fps, P = [], [], []
for s in actives:
    m = Chem.MolFromSmiles(str(s))
    if m is None:
        continue
    active_mols.append(m)
    active_fps.append(gen.GetFingerprint(m))
    P.append(props(m))
P = np.array(P, float)
lo = np.percentile(P, 5, axis=0)
hi = np.percentile(P, 95, axis=0)
lbl = ["MW", "logP", "HBD", "HBA", "RotB", "charge"]
print("active 성질 창(5~95%):")
for i, name in enumerate(lbl):
    print(f"  {name:6s} {lo[i]:8.2f} ~ {hi[i]:8.2f}")

🔎 **코드 뜯어보기 (셀 3)**
- `def props(mol): return (Descriptors.MolWt(mol), Descriptors.MolLogP(mol), ...)` : 분자 하나의 성질 6개를 **튜플**(괄호로 묶은 값 모음)로 반환. `MolWt`=분자량, `MolLogP`=지용성 등.
- `np.percentile(P, 5, axis=0)` / `np.percentile(P, 95, axis=0)` : 각 성질의 **5·95 백분위**(하위/상위 5% 지점) → decoy 매칭 범위 lo~hi.
- `gen.GetFingerprint(m)` : Tanimoto 비교용 지문 계산.

### 셀 4 — PubChem에 요청하는 함수들
PubChem에서 무작위 분자의 SMILES·성질을 받아오고, 성질 범위 안인지 판정하는 함수를 만든다.

In [ ]:
def detect_smiles_prop():
    for name in ("SMILES", "ConnectivitySMILES", "CanonicalSMILES", "IsomericSMILES"):
        try:
            r = requests.get(f"{BASE}/compound/cid/2244/property/{name}/JSON", timeout=20)
            if r.ok and "PropertyTable" in r.json():
                return name
        except Exception:
            pass
    return "CanonicalSMILES"


SPROP = detect_smiles_prop()
print(f"PubChem SMILES 속성명: {SPROP}")

NUM = "MolecularWeight,XLogP,HBondDonorCount,HBondAcceptorCount,RotatableBondCount,Charge"


def fetch_batch(cids):
    cid_str = ",".join(map(str, cids))
    url = f"{BASE}/compound/cid/{cid_str}/property/{NUM},{SPROP}/JSON"
    try:
        r = requests.get(url, timeout=40)
        if not r.ok:
            return []
        return r.json().get("PropertyTable", {}).get("Properties", [])
    except Exception:
        return []


def in_window(p):
    try:
        v = (float(p["MolecularWeight"]), float(p["XLogP"]),
             float(p["HBondDonorCount"]), float(p["HBondAcceptorCount"]),
             float(p["RotatableBondCount"]), float(p["Charge"]))
    except (TypeError, ValueError, KeyError):
        return False
    return all(lo[i] <= v[i] <= hi[i] for i in range(6))

🔎 **코드 뜯어보기 (셀 4)**
- `requests.get(url, timeout=...)` : 주소로 요청 보내기. `.json()`=응답을 파이썬 데이터로. `r.ok`=성공 여부.
- `def in_window(p): ... return all(lo[i] <= v[i] <= hi[i] for i in range(6))` : 성질 6개가 **모두** 범위 안인지. `all(...)`=전부 참이면 True.
- `try/except`로 값이 없거나 변환 실패 시 False 처리.

### 셀 5 — decoy 수집 루프
무작위 화합물을 계속 받아 성질·구조·중복 조건을 통과하면 decoy로 모은다(필요 수까지).

In [ ]:
decoys = []
decoy_set = set()
sampled = 0
t0 = time.time()
while len(decoys) < n_need and sampled < MAX_SAMPLED:
    cids = [random.randint(1, MAX_CID) for _ in range(BATCH)]
    sampled += BATCH
    for p in fetch_batch(cids):
        if len(decoys) >= n_need:
            break
        if not in_window(p):
            continue
        smi = p.get(SPROP)
        if not smi:
            continue
        mol = Chem.MolFromSmiles(str(smi))
        if mol is None:
            continue
        canon = Chem.MolToSmiles(mol)
        if canon in known or canon in decoy_set:
            continue
        sim = max(DataStructs.BulkTanimotoSimilarity(gen.GetFingerprint(mol), active_fps))
        if sim > TANIMOTO_MAX:
            continue
        decoys.append(canon)
        decoy_set.add(canon)
    if sampled % (BATCH * 20) == 0:
        rate = len(decoys) / sampled * 100
        el = time.time() - t0
        print(f"  살펴본 CID {sampled:>7d} | decoy {len(decoys):>5d}/{n_need} "
              f"| 채택률 {rate:.2f}% | {el:.0f}s")
    time.sleep(0.15)   # PubChem 예의상 rate limit

print(f"\ndecoy 수집 완료: {len(decoys)}개 (살펴본 CID {sampled}, {time.time()-t0:.0f}s)")

🔎 **코드 뜯어보기 (셀 5)**
- `while len(decoys) < n_need and sampled < MAX_SAMPLED:` : 필요 수를 채울 때까지(또는 한도까지) **반복**. `and`=두 조건 모두.
- `random.randint(1, MAX_CID)` : 1~최대 사이 **무작위 정수**(랜덤 PubChem 번호).
- `DataStructs.BulkTanimotoSimilarity(fp, active_fps)` : 한 지문과 active 전체의 유사도 목록. `max(...)`=가장 닮은 정도. 0.35 넘으면 탈락.
- `time.sleep(0.15)` : PubChem에 부담 안 주게 잠깐 대기.

### 셀 6 — decoy와 학습셋 저장
모은 decoy와, active+실측inactive+decoy를 합친 학습셋을 저장한다.

In [ ]:
pd.DataFrame({"canonical_smiles": decoys}).to_csv(OUT_CSV, index=False)

train = pd.DataFrame(
    [(s, 1, "real") for s in actives] +
    [(s, 0, "real") for s in inactives] +
    [(s, 0, "decoy") for s in decoys],
    columns=["canonical_smiles", "label", "source"])
train.to_excel(OUT_XLSX, index=False)

print(f"\n최종 학습셋: active {int((train.label==1).sum())} / "
      f"inactive {int((train.label==0).sum())} "
      f"(실측 {len(inactives)} + decoy {len(decoys)})")
print("저장:", OUT_CSV, "|", OUT_XLSX)

🔎 **코드 뜯어보기 (셀 6)**
- `pd.DataFrame({"canonical_smiles": decoys}).to_csv(OUT_CSV, index=False)` : decoy 목록을 CSV로.
- `pd.DataFrame([(s,1,"real") for s in actives] + ...)` : (SMILES, 라벨, 출처) **튜플들의 리스트**로 표를 만들어 학습셋 저장.